[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/03-census-demographics.ipynb)

# Census Demographics

The US Census Bureau collects an extraordinary amount of data about the people and communities that make up the United States. In this notebook, you will learn how to tap into that data using SocialMapper's `get_census_data` function. We will use **Denver, Colorado** as our study area and walk through every step from fetching raw numbers to building visualizations that reveal patterns in population, income, and housing.

By the end of this notebook you will be able to:

1. Explain what the American Community Survey is and why it matters
2. Understand how census variables are named and organized
3. Fetch census data for an isochrone, a list of GEOIDs, or a single point
4. Inspect the `CensusDataResult` object returned by the API
5. Merge demographic data with geographic boundaries
6. Compute aggregate statistics and create publication-quality visualizations

## Setup

We begin by importing the three SocialMapper functions we need, plus `pandas` for tabular data manipulation and `matplotlib` for plotting.

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

In [ ]:
from socialmapper import create_isochrone, get_census_blocks, get_census_data

import matplotlib.pyplot as plt
import pandas as pd

## What Is the American Community Survey?

Before we write any code, it helps to understand *where* the data comes from.

The **American Community Survey (ACS)** is an ongoing survey conducted by the **US Census Bureau**. Unlike the decennial census, which attempts a complete headcount every ten years, the ACS samples approximately **3.5 million households every year**. It collects detailed information about demographics, economics, housing, education, commuting, and more.

Because it is a *sample* rather than a complete count, every ACS value is technically an **estimate** accompanied by a margin of error. The Census Bureau publishes two main products:

| Product | Timeframe | Best for |
|---|---|---|
| **1-year estimates** | Single calendar year | Large geographies (population >= 65,000) |
| **5-year estimates** | Rolling 5-year window | All geographies, including block groups |

SocialMapper uses the **5-year estimates** by default. When you set `year=2023`, you are actually requesting the 2019--2023 pooled estimates. The 5-year product has two major advantages:

1. **Larger effective sample size** -- pooling five years of responses produces more reliable estimates, especially for small geographies.
2. **Full geographic coverage** -- 5-year estimates are available all the way down to the **block group** level (roughly 600--3,000 people), which is the finest resolution the ACS supports.

Keep in mind that because these are estimates, any single block group's value may carry a meaningful margin of error. Aggregating many block groups (as we do inside an isochrone) reduces that uncertainty considerably.

## Understanding Census Variables

Every piece of data in the ACS is identified by a **variable code** -- sometimes called a **B-code** because most table IDs start with the letter B. The naming convention follows a pattern:

```
B19013_001E
^     ^   ^
|     |   +-- Suffix: E = Estimate (M = Margin of Error)
|     +------ Sequence number within the table (001 = first column)
+------------ Table ID: B19013 = "Median Household Income"
```

There are **thousands** of tables and tens of thousands of individual variables. Memorizing codes is impractical, so SocialMapper provides **friendly names** that map to the most commonly used variables. You can pass either the friendly name or the raw B-code to `get_census_data` -- whichever you prefer.

The full mapping is shown in the table below.

| Friendly Name | Census B-Code | Description |
|---|---|---|
| `population` | B01003_001E | Total population |
| `median_income` | B19013_001E | Median household income (dollars) |
| `median_age` | B01002_001E | Median age (years) |
| `housing_units` | B25001_001E | Total housing units |
| `poverty` | B17001_002E | Population below the poverty line |
| `occupied_housing` | B25003_001E | Occupied housing units |
| `owner_occupied` | B25003_002E | Owner-occupied housing units |
| `renter_occupied` | B25003_003E | Renter-occupied housing units |
| `white_population` | B02001_002E | White alone population |
| `black_population` | B02001_003E | Black or African American alone population |
| `asian_population` | B02001_005E | Asian alone population |
| `hispanic_population` | B03002_012E | Hispanic or Latino population (any race) |
| `bachelors_degree` | B15003_022E | Population 25+ with a bachelor's degree |
| `high_school` | B15003_017E | Population 25+ with a high school diploma |
| `households_no_vehicle` | B08201_002E | Households with no vehicle available |
| `median_home_value` | B25077_001E | Median home value (dollars) |
| `median_rent` | B25064_001E | Median gross rent (dollars) |

You can mix friendly names and B-codes in the same request -- SocialMapper resolves them transparently.

## 1. Create an Isochrone for Denver

Our first step is to define a study area. We will create a 15-minute driving isochrone centered on Denver, CO. This polygon represents everywhere a person could drive within 15 minutes of downtown Denver under typical conditions.

In [ ]:
iso = create_isochrone("Denver, CO", travel_time=15, travel_mode="drive")

print(f"Isochrone type:  {iso['type']}")
print(f"Travel time:     {iso['properties']['travel_time']} minutes")
print(f"Travel mode:     {iso['properties']['travel_mode']}")
print(f"Area:            {iso['properties']['area_sq_km']:.1f} sq km")

## 2. Fetch Census Data for the Isochrone

Now we pass the isochrone directly to `get_census_data`. Behind the scenes, SocialMapper:

1. Finds every census block group that intersects the isochrone polygon
2. Queries the Census Bureau API for the requested variables in those block groups
3. Returns a `CensusDataResult` object

Let us start with a single variable -- `population` -- to see the structure of the result.

In [ ]:
census_result = get_census_data(iso, variables=["population"])

print(f"Location type:  {census_result.location_type}")
print(f"Block groups:   {len(census_result.data)}")

## 3. Inspect the CensusDataResult

The `CensusDataResult` is a Pydantic model with three fields:

| Field | Type | Description |
|---|---|---|
| `data` | `dict[str, dict[str, Any]]` | Nested dictionary keyed by GEOID. Each value is a dictionary mapping variable names to numeric values. |
| `location_type` | `"polygon"`, `"geoids"`, or `"point"` | Tells you what kind of location was used in the query. |
| `query_info` | `dict[str, Any]` | Metadata about the query -- the year, variable names requested, resolved census codes, etc. |

Let us explore each field.

In [ ]:
# query_info tells us exactly what was sent to the Census API
for key, value in census_result.query_info.items():
    print(f"{key}: {value}")

In [ ]:
# The data dict is keyed by 12-digit GEOIDs
# Each GEOID maps to a dict of {variable_name: value}
first_geoid = list(census_result.data.keys())[0]
print(f"GEOID:  {first_geoid}")
print(f"Data:   {census_result.data[first_geoid]}")
print()
print("A GEOID encodes geography hierarchically:")
print(f"  State FIPS:    {first_geoid[:2]}")
print(f"  County FIPS:   {first_geoid[2:5]}")
print(f"  Census tract:  {first_geoid[5:11]}")
print(f"  Block group:   {first_geoid[11]}")

## 4. Fetch Multiple Variables at Once

In practice you will almost always want more than one variable. Pass a list of friendly names and SocialMapper resolves each to the correct B-code, batches them into efficient Census API calls, and returns everything in a single result.

In [ ]:
multi = get_census_data(
    iso,
    variables=["population", "median_income", "median_age", "housing_units", "poverty"]
)

print(f"Friendly names requested: {multi.query_info['variables']}")
print(f"Census codes resolved:    {multi.query_info['variable_codes']}")
print(f"Block groups returned:    {len(multi.data)}")
print()

# Preview the first three block groups
for geoid in list(multi.data.keys())[:3]:
    print(f"{geoid}: {multi.data[geoid]}")

## 5. Using Raw Census B-Codes

If you already know the exact B-code you need -- perhaps from the Census Bureau's data explorer or an academic paper -- you can pass it directly. This is useful when you want a variable that SocialMapper does not have a friendly name for.

In [ ]:
raw = get_census_data(iso, variables=["B01003_001E", "B19013_001E"])

sample_geoid = list(raw.data.keys())[0]
print(f"GEOID: {sample_geoid}")
print(f"Data:  {raw.data[sample_geoid]}")
print()
print("Notice the keys are the raw B-codes since we did not use friendly names.")

## 6. Query by GEOID List

Sometimes you already know exactly which block groups you care about. Perhaps you received a list of GEOIDs from a colleague, a database, or a prior analysis. In that case, pass a plain Python list of 12-digit GEOID strings as the `location` argument.

We will grab a few GEOIDs from our earlier isochrone to demonstrate.

In [ ]:
blocks = get_census_blocks(polygon=iso)
geoid_list = [b["geoid"] for b in blocks[:5]]
print(f"Querying GEOIDs: {geoid_list}")

geoid_result = get_census_data(geoid_list, variables=["population", "median_income"])
print(f"\nLocation type: {geoid_result.location_type}")
for geoid, data in geoid_result.data.items():
    print(f"  {geoid}: {data}")

## 7. Query by Point Location

For the most targeted query, pass a `(latitude, longitude)` tuple. SocialMapper identifies the single block group that contains that point and returns its data. This is useful when you want demographics for a specific address or coordinate.

In [ ]:
# Colorado State Capitol building
point_result = get_census_data((39.7392, -104.9903), variables=["population", "median_income"])

print(f"Location type:       {point_result.location_type}")
print(f"Block groups found:  {len(point_result.data)}")
for geoid, data in point_result.data.items():
    print(f"  {geoid}: {data}")

## 8. Merging Demographics with Block Geometries

Census data on its own is just numbers keyed by GEOIDs. To put those numbers on a map, you need the **geographic boundaries** of each block group. That is what `get_census_blocks` provides.

The merge step connects the two by matching on the GEOID field that both datasets share:

```
get_census_blocks(polygon=iso)  -->  [{"geoid": "080310001001", "geometry": {...}, ...}, ...]
get_census_data(iso, vars)      -->  {"080310001001": {"population": 2451, ...}, ...}
                                          ^
                                     join key
```

After merging, each record contains both the polygon geometry *and* the demographic values -- exactly what a choropleth map needs (covered in Notebook 04).

In [ ]:
# Fetch block geometries and a rich set of census variables
blocks = get_census_blocks(polygon=iso)
census = get_census_data(
    iso,
    variables=[
        "population", "median_income", "median_age",
        "housing_units", "poverty", "renter_occupied", "owner_occupied"
    ]
)

# Merge by GEOID
merged = []
for block in blocks:
    geoid = block["geoid"]
    if geoid in census.data:
        entry = {**block, **census.data[geoid]}
        merged.append(entry)

print(f"Block groups with geometry:      {len(blocks)}")
print(f"Block groups with census data:   {len(census.data)}")
print(f"Successfully merged:             {len(merged)}")
print(f"\nKeys in a merged record:")
print(f"  {list(merged[0].keys())}")

## 9. Building a DataFrame for Analysis

With the data merged, we can load everything into a pandas DataFrame. This makes it easy to compute statistics, filter, sort, and feed into matplotlib for visualization.

In [ ]:
df = pd.DataFrame(merged)

# Drop the geometry column for cleaner display (we still have it in 'merged')
display_cols = ["geoid", "population", "median_income", "median_age",
                "housing_units", "poverty", "owner_occupied", "renter_occupied"]
df[display_cols].head(10)

## 10. Aggregate Statistics

Aggregating across all block groups gives us a profile of the entire study area. Some variables, like population, are counts that should be **summed**. Others, like median income, are themselves medians -- taking the mean of medians gives a rough approximation, but it is not the true area median. For rigorous analysis, you would weight by population, but for exploratory work the simple statistics below are informative.

In [ ]:
# Convert to numeric, coercing any non-numeric values to NaN
numeric_cols = ["population", "median_income", "median_age",
                "housing_units", "poverty", "owner_occupied", "renter_occupied"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("=" * 50)
print("  Denver 15-Minute Drive Area -- Summary")
print("=" * 50)
print(f"  Block groups:          {len(df):,}")
print(f"  Total population:      {df['population'].sum():,.0f}")
print(f"  Avg population / BG:   {df['population'].mean():,.0f}")
print(f"  Total housing units:   {df['housing_units'].sum():,.0f}")
print(f"  Total in poverty:      {df['poverty'].sum():,.0f}")
print()
print("  Median Income Across Block Groups")
print(f"    Min:    ${df['median_income'].min():>10,.0f}")
print(f"    25th:   ${df['median_income'].quantile(0.25):>10,.0f}")
print(f"    Median: ${df['median_income'].median():>10,.0f}")
print(f"    75th:   ${df['median_income'].quantile(0.75):>10,.0f}")
print(f"    Max:    ${df['median_income'].max():>10,.0f}")
print()
print("  Median Age Across Block Groups")
print(f"    Min:    {df['median_age'].min():>6.1f} years")
print(f"    Median: {df['median_age'].median():>6.1f} years")
print(f"    Max:    {df['median_age'].max():>6.1f} years")
print("=" * 50)

## 11. Histogram: Population Distribution Across Block Groups

A histogram shows how block group populations are distributed. Most urban block groups contain between 500 and 3,000 people by design, but you may see outliers -- very dense apartment clusters or sparsely populated industrial areas.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

pop_data = df["population"].dropna()
ax.hist(pop_data, bins=30, color="#5a9bd5", edgecolor="white", linewidth=0.6)
ax.axvline(pop_data.median(), color="#d63384", linewidth=2, linestyle="--",
           label=f"Median: {pop_data.median():,.0f}")
ax.set_xlabel("Population per Block Group", fontsize=12)
ax.set_ylabel("Number of Block Groups", fontsize=12)
ax.set_title("Population Distribution -- Denver 15-min Drive Area",
             fontsize=14, fontweight="bold", color="#1b2a4a")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 12. Histogram: Median Income Distribution

Income distributions in urban areas are often right-skewed: many block groups cluster around a moderate income level, with a long tail of wealthier neighborhoods. A vertical line at the median helps anchor your interpretation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

income_data = df["median_income"].dropna()
# Filter out sentinel values (Census uses large negatives or 0 for missing data)
income_data = income_data[income_data > 0]

ax.hist(income_data, bins=30, color="#66b266", edgecolor="white", linewidth=0.6)
ax.axvline(income_data.median(), color="#d63384", linewidth=2, linestyle="--",
           label=f"Median: ${income_data.median():,.0f}")
ax.set_xlabel("Median Household Income ($)", fontsize=12)
ax.set_ylabel("Number of Block Groups", fontsize=12)
ax.set_title("Median Income Distribution -- Denver 15-min Drive Area",
             fontsize=14, fontweight="bold", color="#1b2a4a")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)

# Format x-axis as dollars
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()

## 13. Scatter Plot: Population vs. Median Income

Is there a relationship between how many people live in a block group and the income level of that block group? A scatter plot lets us see the joint distribution. In many metros, the highest-income block groups tend to have *moderate* populations -- very dense areas often have lower median incomes, while the wealthiest neighborhoods tend to be suburban with fewer but larger homes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Use only rows with valid data
scatter_df = df[["population", "median_income"]].dropna()
scatter_df = scatter_df[(scatter_df["population"] > 0) & (scatter_df["median_income"] > 0)]

ax.scatter(
    scatter_df["population"],
    scatter_df["median_income"],
    alpha=0.5, s=40, color="#5a9bd5", edgecolor="#2a6099", linewidth=0.4
)
ax.set_xlabel("Population", fontsize=12)
ax.set_ylabel("Median Household Income ($)", fontsize=12)
ax.set_title("Population vs. Median Income by Block Group",
             fontsize=14, fontweight="bold", color="#1b2a4a")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 14. Box Plots: Comparing Variables Side by Side

Box plots are excellent for comparing the spread and central tendency of different variables. Because our variables have very different scales (population counts vs. dollar amounts vs. years), we will use separate subplots rather than trying to normalize them onto one axis. Each subplot shows the interquartile range (box), median (line), and outliers (dots).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

box_configs = [
    ("population", "Population", "#5a9bd5"),
    ("median_income", "Median Income ($)", "#66b266"),
    ("median_age", "Median Age (years)", "#e8913a"),
    ("poverty", "Poverty Count", "#c45a5a"),
]

for ax, (col, label, color) in zip(axes, box_configs):
    col_data = df[col].dropna()
    col_data = col_data[col_data > 0] if col in ("median_income",) else col_data
    bp = ax.boxplot(
        col_data, vert=True, patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.7),
        medianprops=dict(color="#1b2a4a", linewidth=2),
        flierprops=dict(marker="o", markersize=3, alpha=0.4),
        whiskerprops=dict(color="#666666"),
        capprops=dict(color="#666666")
    )
    ax.set_title(label, fontsize=11, fontweight="bold", color="#1b2a4a")
    ax.set_xticks([])
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Variable Distributions Across Block Groups -- Denver 15-min Drive Area",
             fontsize=14, fontweight="bold", color="#1b2a4a", y=1.02)
plt.tight_layout()
plt.show()

## 15. Housing Tenure: Owner vs. Renter Occupied

Understanding the split between owner-occupied and renter-occupied housing can reveal a lot about a neighborhood's character. Areas with high renter proportions often correspond to denser, more urban cores with apartment buildings, while owner-dominated areas tend to be more suburban.

In [ ]:
tenure_df = df[["owner_occupied", "renter_occupied"]].dropna()
total_owner = tenure_df["owner_occupied"].sum()
total_renter = tenure_df["renter_occupied"].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: bar chart of totals
categories = ["Owner-Occupied", "Renter-Occupied"]
values = [total_owner, total_renter]
colors = ["#5a9bd5", "#e8913a"]
bars = ax1.bar(categories, values, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
             f"{val:,.0f}", ha="center", fontsize=12, fontweight="bold", color="#1b2a4a")
ax1.set_ylabel("Housing Units", fontsize=12)
ax1.set_title("Housing Tenure Totals", fontsize=13, fontweight="bold", color="#1b2a4a")
ax1.spines[["top", "right"]].set_visible(False)

# Right: histogram of renter fraction per block group
tenure_df = tenure_df.copy()
tenure_df["total"] = tenure_df["owner_occupied"] + tenure_df["renter_occupied"]
tenure_df = tenure_df[tenure_df["total"] > 0]
tenure_df["renter_pct"] = tenure_df["renter_occupied"] / tenure_df["total"] * 100

ax2.hist(tenure_df["renter_pct"], bins=25, color="#e8913a", edgecolor="white", linewidth=0.6)
ax2.axvline(50, color="#666666", linewidth=1, linestyle=":", label="50% line")
ax2.set_xlabel("Renter-Occupied Percentage (%)", fontsize=12)
ax2.set_ylabel("Number of Block Groups", fontsize=12)
ax2.set_title("Renter Fraction Distribution", fontsize=13, fontweight="bold", color="#1b2a4a")
ax2.legend(fontsize=10)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## Summary

### Key Concepts

- The **American Community Survey** is the primary source of detailed US demographic data between decennial censuses. It surveys ~3.5 million households per year.
- **5-year estimates** (e.g., 2019--2023 for `year=2023`) pool five years of responses to provide reliable estimates at fine geographic levels, including **block groups**.
- ACS variables follow a **B-code** naming convention: table ID + sequence number + suffix (E for estimate, M for margin of error).
- SocialMapper provides **friendly names** (like `population` and `median_income`) as convenient aliases for the most commonly used B-codes.
- The **merge step** -- joining census data to block group geometries by GEOID -- is the bridge between raw numbers and spatial visualization.

### API Quick Reference

| Task | Code |
|---|---|
| Census data from an isochrone | `get_census_data(iso, ["population"])` |
| Multiple variables | `get_census_data(iso, ["population", "median_income", "median_age"])` |
| Friendly names or B-codes | `"median_income"` or `"B19013_001E"` |
| Query by GEOID list | `get_census_data(["080310001001", ...], variables=[...])` |
| Query by point | `get_census_data((39.7392, -104.9903), variables=[...])` |
| Get block geometries | `get_census_blocks(polygon=iso)` |
| Merge data + geometry | Loop over blocks, match by GEOID |

### What's Next?

Now that you can fetch and explore census data, the natural next step is to **put it on a map**. In the next notebook we build choropleth maps that color each block group by a demographic variable, making spatial patterns immediately visible.

[04 -- Choropleth Maps](04-choropleth-maps.ipynb)